# Assignment 02: Backpropagation by Hand

**Module:** 04 — Neural Networks Deep Dive  
**Estimated Time:** 6-8 hours

---

## Learning Objectives

- Compute the full forward and backward pass of a 3-layer network by hand with specific numbers
- Verify every gradient against PyTorch autograd
- Compute backpropagation through BatchNorm
- Understand why backpropagation is computationally efficient

### Network Architecture

- Input: $\mathbf{x} \in \mathbb{R}^2$
- Hidden 1: 3 neurons, ReLU
- Hidden 2: 2 neurons, ReLU
- Output: 2 neurons, softmax
- Loss: cross-entropy

### The Weights

$$\mathbf{W}_1 = \begin{bmatrix} 0.1 & 0.2 \\ -0.3 & 0.4 \\ 0.5 & -0.1 \end{bmatrix}, \quad \mathbf{b}_1 = \begin{bmatrix} 0.1 \\ -0.2 \\ 0.0 \end{bmatrix}$$

$$\mathbf{W}_2 = \begin{bmatrix} 0.3 & -0.2 & 0.1 \\ 0.4 & 0.1 & -0.3 \end{bmatrix}, \quad \mathbf{b}_2 = \begin{bmatrix} 0.0 \\ 0.1 \end{bmatrix}$$

$$\mathbf{W}_3 = \begin{bmatrix} 0.2 & 0.5 \\ -0.1 & 0.3 \end{bmatrix}, \quad \mathbf{b}_3 = \begin{bmatrix} 0.1 \\ -0.1 \end{bmatrix}$$

### Input

$$\mathbf{x} = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}, \quad \mathbf{y}_{true} = \begin{bmatrix} 1 \\ 0 \end{bmatrix} \text{ (class 0)}$$

In [ ]:
import sys
import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, '../../')
from shared_utils.common import set_seed

set_seed(42)
print('Setup complete.')

---
## Part 1: Forward Pass by Hand

Compute every intermediate value. Show all work.

**Example for $z_1[0]$:**
$$z_1[0] = W_1[0,0] \cdot x[0] + W_1[0,1] \cdot x[1] + b_1[0] = 0.1 \times 1.0 + 0.2 \times 2.0 + 0.1 = 0.6$$

In [ ]:
# --- Define weights ---
W1 = np.array([[ 0.1,  0.2],
               [-0.3,  0.4],
               [ 0.5, -0.1]])
b1 = np.array([0.1, -0.2, 0.0])

W2 = np.array([[ 0.3, -0.2,  0.1],
               [ 0.4,  0.1, -0.3]])
b2 = np.array([0.0, 0.1])

W3 = np.array([[ 0.2,  0.5],
               [-0.1,  0.3]])
b3 = np.array([0.1, -0.1])

x = np.array([1.0, 2.0])
y_true = np.array([1.0, 0.0])  # class 0, one-hot

In [ ]:
# --- Layer 1: z1 = W1 @ x + b1 ---
z1 = W1 @ x + b1
print(f'z1 = {z1}')
# YOUR HAND COMPUTATION:
# z1[0] = 0.1*1.0 + 0.2*2.0 + 0.1 = 0.6
# z1[1] = -0.3*1.0 + 0.4*2.0 + (-0.2) = ?
# z1[2] = 0.5*1.0 + (-0.1)*2.0 + 0.0 = ?

In [ ]:
# --- Layer 1 activation: a1 = relu(z1) ---
a1 = np.maximum(0, z1)
print(f'a1 = {a1}')
# Note which neurons are active (z > 0)

In [ ]:
# --- Layer 2: z2 = W2 @ a1 + b2 ---
z2 = W2 @ a1 + b2
print(f'z2 = {z2}')
# YOUR HAND COMPUTATION for each element

In [ ]:
# --- Layer 2 activation: a2 = relu(z2) ---
a2 = np.maximum(0, z2)
print(f'a2 = {a2}')

In [ ]:
# --- Output: z3 = W3 @ a2 + b3 ---
z3 = W3 @ a2 + b3
print(f'z3 = {z3}')

In [ ]:
# --- Softmax ---
exp_z3 = np.exp(z3 - np.max(z3))  # numerical stability
y_hat = exp_z3 / exp_z3.sum()
print(f'y_hat = {y_hat}')
print(f'Sum = {y_hat.sum():.6f} (should be 1.0)')

In [ ]:
# --- Cross-entropy loss ---
L = -np.sum(y_true * np.log(y_hat + 1e-8))
print(f'Loss L = {L:.6f}')

---
## Part 2: Backward Pass by Hand

### Step 1: Output Layer Gradient

$\frac{\partial L}{\partial \mathbf{z}_3} = \hat{\mathbf{y}} - \mathbf{y}_{true}$

In [ ]:
# --- Step 1: dL/dz3 ---
dL_dz3 = y_hat - y_true
print(f'dL/dz3 = {dL_dz3}')

In [ ]:
# --- Step 2: dL/dW3 and dL/db3 ---
# dL/dW3 = dL/dz3 (outer) a2
dL_dW3 = np.outer(dL_dz3, a2)
dL_db3 = dL_dz3.copy()

print(f'dL/dW3 =\n{dL_dW3}')
print(f'dL/db3 = {dL_db3}')
# YOUR HAND COMPUTATION: write out each element

In [ ]:
# --- Step 3: Propagate to layer 2 ---
# dL/da2 = W3^T @ dL/dz3
dL_da2 = W3.T @ dL_dz3
print(f'dL/da2 = {dL_da2}')

# dL/dz2 = dL/da2 * relu'(z2)
relu_mask_z2 = (z2 > 0).astype(float)
dL_dz2 = dL_da2 * relu_mask_z2
print(f'relu_mask_z2 = {relu_mask_z2}')
print(f'dL/dz2 = {dL_dz2}')

In [ ]:
# --- Step 4: dL/dW2 and dL/db2 ---
dL_dW2 = np.outer(dL_dz2, a1)
dL_db2 = dL_dz2.copy()

print(f'dL/dW2 =\n{dL_dW2}')
print(f'dL/db2 = {dL_db2}')

In [ ]:
# --- Step 5: Propagate to layer 1 ---
dL_da1 = W2.T @ dL_dz2
relu_mask_z1 = (z1 > 0).astype(float)
dL_dz1 = dL_da1 * relu_mask_z1

print(f'dL/da1 = {dL_da1}')
print(f'relu_mask_z1 = {relu_mask_z1}')
print(f'dL/dz1 = {dL_dz1}')

In [ ]:
# --- Step 6: dL/dW1 and dL/db1 ---
dL_dW1 = np.outer(dL_dz1, x)
dL_db1 = dL_dz1.copy()

print(f'dL/dW1 =\n{dL_dW1}')
print(f'dL/db1 = {dL_db1}')

### Summary Table

| Parameter | Gradient |
|-----------|----------|
| W3 | (fill in) |
| b3 | (fill in) |
| W2 | (fill in) |
| b2 | (fill in) |
| W1 | (fill in) |
| b1 | (fill in) |

---
## Part 3: Verification with PyTorch

Create the exact same network and verify every gradient.

In [ ]:
# --- PyTorch verification ---
# Note: nn.Linear(in, out) stores weight as (out, in) and
# computes output = input @ weight.T + bias
# So input should be (batch, features)

# Layer 1: 2 -> 3
layer1 = nn.Linear(2, 3)
layer1.weight.data = torch.tensor(W1, dtype=torch.float64)
layer1.bias.data = torch.tensor(b1, dtype=torch.float64)

# Layer 2: 3 -> 2
layer2 = nn.Linear(3, 2)
layer2.weight.data = torch.tensor(W2, dtype=torch.float64)
layer2.bias.data = torch.tensor(b2, dtype=torch.float64)

# Layer 3: 2 -> 2
layer3 = nn.Linear(2, 2)
layer3.weight.data = torch.tensor(W3, dtype=torch.float64)
layer3.bias.data = torch.tensor(b3, dtype=torch.float64)

# Forward pass
x_t = torch.tensor(x, dtype=torch.float64).unsqueeze(0)  # (1, 2)
y_t = torch.tensor([0], dtype=torch.long)  # class 0

z1_t = layer1(x_t)
a1_t = torch.relu(z1_t)
z2_t = layer2(a1_t)
a2_t = torch.relu(z2_t)
z3_t = layer3(a2_t)

# Cross-entropy loss (combines softmax + CE)
loss_t = nn.functional.cross_entropy(z3_t, y_t)

print(f'PyTorch loss = {loss_t.item():.6f}')
print(f'NumPy loss   = {L:.6f}')
assert abs(loss_t.item() - L) < 1e-5, 'Loss mismatch!'

# Backward
loss_t.backward()

In [ ]:
# --- Compare all gradients ---
print('=== GRADIENT COMPARISON ===')

comparisons = [
    ('W3', dL_dW3, layer3.weight.grad.numpy()),
    ('b3', dL_db3, layer3.bias.grad.numpy()),
    ('W2', dL_dW2, layer2.weight.grad.numpy()),
    ('b2', dL_db2, layer2.bias.grad.numpy()),
    ('W1', dL_dW1, layer1.weight.grad.numpy()),
    ('b1', dL_db1, layer1.bias.grad.numpy()),
]

for name, hand, pytorch in comparisons:
    max_diff = np.abs(hand - pytorch).max()
    status = 'PASS' if max_diff < 1e-5 else 'FAIL'
    print(f'dL/d{name}: max diff = {max_diff:.2e} [{status}]')
    if max_diff >= 1e-5:
        print(f'  Hand:    {hand}')
        print(f'  PyTorch: {pytorch}')

---
## Part 4: Backpropagation Through BatchNorm

Add BatchNorm after each hidden layer and repeat with a batch of 2 samples.

In [ ]:
# --- Part 4: BatchNorm ---
x_batch = np.array([[1.0, 2.0],
                     [1.5, 0.5]])
y_batch_labels = np.array([0, 1])

gamma1 = np.array([1.0, 1.0, 1.0])
beta1 = np.array([0.0, 0.0, 0.0])
eps = 1e-5

# YOUR CODE HERE
# 1. Forward pass through layer 1 for both samples
# 2. Apply BatchNorm:
#    mu = mean over batch
#    var = variance over batch
#    z_hat = (z - mu) / sqrt(var + eps)
#    z_bn = gamma * z_hat + beta
# 3. Continue forward pass
# 4. Verify with PyTorch (using nn.BatchNorm1d)
pass

In [ ]:
# --- BatchNorm backward (at least for layer 1) ---
# YOUR CODE HERE
# dL/dgamma_j = sum_i(dL/dz_bn_j^(i) * z_hat_j^(i))
# dL/dbeta_j = sum_i(dL/dz_bn_j^(i))
# dL/dz_j^(i) = (gamma / sqrt(var+eps)) * (dL/dz_bn - mean(dL/dbeta)/m
#               - z_hat * mean(dL/dgamma)/m)
# Verify with PyTorch
pass

---
## Part 5: Essay — Why Backpropagation Is Efficient

Write 500-800 words addressing:

1. **The naive alternative**: Numerical differentiation requires $O(N)$ forward passes per gradient. For 1M parameters = 1M forward passes.

2. **Backprop's complexity**: 1 forward + 1 backward pass ($\sim 2\times$ forward cost). Total: $O(1)$ forward passes regardless of $N$.

3. **The ratio**: Backprop is $O(N)$ times faster. For billions of parameters: feasible vs impossible.

4. **Why it works**: Chain rule reuses intermediate computations. $\frac{\partial L}{\partial \mathbf{z}_2}$ computed once, used for both $\frac{\partial L}{\partial \mathbf{W}_2}$ and $\frac{\partial L}{\partial \mathbf{W}_1}$.

5. **Forward vs reverse mode**: Forward mode is $O(N)$ for each output. Reverse mode is $O(1)$ for all parameters given one scalar loss.

*YOUR ESSAY HERE*